In [1]:
import pandas as pd
import duckdb
import numpy as np
import streamlit
import subprocess

In [2]:
conn = duckdb.connect()

In [3]:
df_errors = pd.read_csv(r"data/PdM_errors.csv")
df_failures = pd.read_csv(r"data/PdM_failures.csv")
df_machines = pd.read_csv(r"data/PdM_machines.csv")
df_maint = pd.read_csv(r"data/PdM_maint.csv")
df_telemetry = pd.read_csv(r"data/PdM_telemetry.csv")

In [4]:
# Data Exploration of all the data sources
dfs = {
    'telemetry': df_telemetry,
    'errors':    df_errors,
    'failures':  df_failures,
    'maint':     df_maint,
    'machines':  df_machines
}

for name, df in dfs.items():
    print(f"{'='*50}")
    print(f"{name.upper()} — {df.shape[0]:,} rows x {df.shape[1]} columns")
    print(f"{'='*50}")
    print(df.dtypes)
    print(f"\nnull counts:")
    print(df.isnull().sum())
    print(f"\nsample:")
    print(df.head(3))
    print()

TELEMETRY — 876,100 rows x 6 columns
datetime         str
machineID      int64
volt         float64
rotate       float64
pressure     float64
vibration    float64
dtype: object

null counts:
datetime     0
machineID    0
volt         0
rotate       0
pressure     0
vibration    0
dtype: int64

sample:
              datetime  machineID        volt      rotate    pressure  \
0  2015-01-01 06:00:00          1  176.217853  418.504078  113.077935   
1  2015-01-01 07:00:00          1  162.879223  402.747490   95.460525   
2  2015-01-01 08:00:00          1  170.989902  527.349825   75.237905   

   vibration  
0  45.087686  
1  43.413973  
2  34.178847  

ERRORS — 3,919 rows x 3 columns
datetime       str
machineID    int64
errorID        str
dtype: object

null counts:
datetime     0
machineID    0
errorID      0
dtype: int64

sample:
              datetime  machineID errorID
0  2015-01-03 07:00:00          1  error1
1  2015-01-03 20:00:00          1  error3
2  2015-01-04 06:00:00          1

In [5]:
# Changing field datetime from str to datetime 
dfs = [df_errors, df_failures, df_maint, df_telemetry]

for df in dfs:
    if 'datetime' in df.columns:
        df['datetime'] = pd.to_datetime(df['datetime'])

## Part 1 — SQL Queries
### Q1 — Most recent sensor readings per machine

For each machine, return its most recent telemetry reading , which is one row per machine showing the
latest voltage, rotation, pressure, and vibration values.
Expected output: 100 rows, one per machine, with four sensor value columns.

**What it does:** returns the most recent sensor reading for each of the 100 machines.

**Why this approach:** rank() partitioned by machineID ordered by datetime descending gives rnk = 1 for the latest row per machine. I used rank() over row_number() because if two readings share the exact same timestamp, rank() surfaces both rather than arbitrarily dropping one.

**Edge cases:** no nulls expected in telemetry datetime, if there were, they'd sort last and not affect the result.

In [6]:
query = """ with machine_rnk as 
                    (select machineid , volt , rotate , pressure , vibration, 
                    rank() over(partition by machineid order by datetime desc) as rnk 
                    from df_telemetry
                    )
                    select machineid ,volt, rotate, pressure, vibration
                    from machine_rnk 
                    where rnk = 1 
                    order by machineid
 """

q1 = conn.execute(query).fetch_df()
q1.head(20).style.hide(axis="index")


machineID,volt,rotate,pressure,vibration
1,191.873171,382.736626,100.893691,37.940220
2,182.021908,392.270187,99.946293,41.676184
3,164.906466,410.742075,91.721593,35.476970
4,149.409696,522.123117,112.680499,45.637870
5,178.789197,415.167298,142.414273,47.020210
6,181.209176,393.853197,98.743578,41.879120
7,181.343113,339.771881,107.488234,37.933208
8,168.439184,438.340305,89.904141,32.154524
9,176.007480,414.778970,93.243580,37.594398
10,154.003011,362.335964,111.683489,41.883334


Q2 Pre-error sensor conditions
For every error event in the errors table, compute the average pressure and average vibration
recorded for that machine in the 6-hour window immediately before the error occurred. Include
the machine_id, error type, and the two averages in your output.
If a machine has no telemetry in that 6-hour window, decide what to return and explain your choice.

**What it does:** returns average pressure and vibration for each machine and error type, computed from telemetry in the 6-hour window immediately before each error event.

**Why this approach:** inner join on machineID with a time range condition keeps only error events that have matching telemetry in the window. I chose inner join over left join deliberately, returning null averages for machines with no telemetry in that window would be misleading, so I'd rather exclude those rows and note the gap.

**Edge cases:** if a machine has no telemetry in the 6-hour window before an error, it is excluded from the output entirely rather than returning nulls.

In [7]:
query = """ select de.machineid,
                    de.errorid , 
                    avg(pressure) as avg_pressure,
                    avg(vibration) as avg_vibration 
                    from df_errors de
                    join df_telemetry dm
                    on de.machineid = dm.machineid
                    and CAST(dm.datetime AS TIMESTAMP) >= CAST(de.datetime AS TIMESTAMP) - INTERVAL '6 hour'
                    and cast(dm.datetime as timestamp) < cast(de.datetime as timestamp)
                    group by de.machineid , de.errorid
                    order by de.machineid, de.errorid                    

 """

q2 = conn.execute(query).fetch_df()
q2.head(20).style.hide(axis="index")


machineID,errorID,avg_pressure,avg_vibration
1,error1,100.184193,40.533461
1,error2,100.226839,40.437147
1,error3,101.580889,40.601287
1,error4,100.208975,39.393399
1,error5,102.278852,52.614113
2,error1,98.764961,38.408715
2,error2,103.528189,40.901737
2,error3,100.384489,42.108346
2,error4,98.951310,38.528455
2,error5,103.519294,40.120485


Q3 Maintenance intervals by component
For each machine and component (comp1–comp4), compute the number of days between each
maintenance event and the one immediately before it. Expected output columns: machine_id,
component, maintenance_date, previous_maintenance_date, days_since_last_service. Sort by
machine_id, component, maintenance_date.

**What it does:** returns the number of days between consecutive maintenance events for each machine and component combination.

**Why this approach:** lag() partitioned by machineID and comp ordered by datetime pulls the previous maintenance date for the same machine and component. date_diff('day') then gives the interval. Partitioning by both machineID and comp ensures intervals are calculated within the same component type and not across different ones.

**Edge cases:** the first maintenance event per machine per component has no prior event, so previous_maintenance_date and days_since_last_service are null, which is expected and left as-is.

In [8]:
query = """ with previous_maint_date as 
                    ( select machineid , 
                    comp , 
                    datetime , 
                    lag(datetime) over (partition by machineid, comp order by datetime) as prev_maint_date
                    from df_maint
                    )
                    select machineid, 
                    comp,
                    datetime as maintenance_date,
                    prev_maint_date as previous_maintenance_date,
                    date_diff('day',prev_maint_date,datetime) as days_since_last_service
                    from previous_maint_date
                    order by machineid, comp , datetime

"""
q3 = conn.execute(query).fetch_df()
q3.head(20).style.hide(axis="index")


machineID,comp,maintenance_date,previous_maintenance_date,days_since_last_service
1,comp1,2014-12-13 06:00:00,NaT,
1,comp1,2015-01-05 06:00:00,2014-12-13 06:00:00,23
1,comp1,2015-01-20 06:00:00,2015-01-05 06:00:00,15
1,comp1,2015-03-06 06:00:00,2015-01-20 06:00:00,45
1,comp1,2015-03-21 06:00:00,2015-03-06 06:00:00,15
1,comp1,2015-06-19 06:00:00,2015-03-21 06:00:00,90
1,comp1,2015-07-19 06:00:00,2015-06-19 06:00:00,30
1,comp1,2015-08-03 06:00:00,2015-07-19 06:00:00,15
1,comp1,2015-09-02 06:00:00,2015-08-03 06:00:00,30
1,comp1,2015-10-02 06:00:00,2015-09-02 06:00:00,30


Q4 Failure rate by machine model and age cohort
Compute the failure rate — expressed as failures per 100 machine-hours of operation — for
each combination of machine model and age cohort. Define age cohorts as: 0–5 years, 6–10
years, 11–20 years. Explain how you computed the 'machine-hours of operation' denominator
and why.

**What it does:** returns failure rate expressed as failures per 100 machine-hours for each combination of machine model and age cohort.

**Why this approach:** I defined machine-hours of operation as the count of telemetry rows per model and age cohort, since telemetry is recorded hourly per machine, each row represents one machine-hour. This is the most honest denominator because it reflects actual observed operating time rather than a theoretical calendar estimate. Age cohorts are bucketed with a case statement on the machines table before joining to failures.

**Edge cases:** machines with no failures still appear in the output with a failure rate of zero using a left join from model cohorts to failures. Division is wrapped in nullif to guard against any cohort with zero machine-hours.

In [9]:
query = """ with machine_model_cohort as 
                    ( 
                    select machineid , 
                         model , 
                         case when age between 0 and 5 then '0-5 years'
                              when age between 6 and 10 then '6-10 years'
                              when age between 11 and 20 then '11-20 years'
                              else 'other'
                         end as age_cohort
                         from df_machines
                    ),
                    machine_hours as 
                    (
                         select model, 
                         age_cohort , 
                         count(*) as total_machine_hours
                         from df_telemetry dt
                         join machine_model_cohort mmc 
                         on dt.machineid = mmc.machineid
                         group by model , age_cohort 
                    ),
                    fail_counts as 
                    (
                         select model,
                         age_cohort,
                         count(*) as total_failures
                         from df_failures df
                         join machine_model_cohort mmc 
                         on df.machineid = mmc.machineid
                         group by model, age_cohort
                    )
                    select mh.model, 
                    mh.age_cohort,
                    mh.total_machine_hours,
                    coalesce(fc.total_failures,0) as total_failures,
                    round(coalesce(fc.total_failures,0)*100.0/NULLIF(mh.total_machine_hours,0),3) as failure_per_100_machine_hours
                    from machine_hours mh
                    left join fail_counts fc
                    on mh.model = fc.model
                    and mh.age_cohort = fc.age_cohort
                    order by mh.model , mh.age_cohort
                    
"""
q4 = conn.execute(query).fetch_df()
q4.head(20).style.hide(axis="index")


model,age_cohort,total_machine_hours,total_failures,failure_per_100_machine_hours
model1,0-5 years,26283,30,0.114000
model1,11-20 years,96371,139,0.144000
model1,6-10 years,17522,20,0.114000
model2,0-5 years,17522,21,0.120000
model2,11-20 years,87610,112,0.128000
model2,6-10 years,43805,35,0.080000
model3,0-5 years,43805,27,0.062000
model3,11-20 years,175220,148,0.084000
model3,6-10 years,87610,46,0.053000
model4,0-5 years,96371,46,0.048000


Q5 Top error codes before each failure
For each confirmed failure event, identify the three most frequently occurring error codes logged
for that machine in the 24 hours preceding the failure. Expected output: one row per failure, with
columns for failure_id, machine_id, failure_component, and three columns for the top-ranked
error codes (rank_1_error, rank_2_error, rank_3_error). If fewer than three distinct error codes
exist in that window, NULL-fill the remaining columns.

**What it does:** returns the three most frequently occurring error codes in the 24 hours before each confirmed failure, one row per failure with rank_1_error, rank_2_error, rank_3_error columns.

**Why this approach:** for each failure I join errors on machineID within the 24-hour window, count occurrences per error code, then rank them by count descending using row_number(). The top 3 ranks are pivoted into columns using max(case when rank = N). I used a left join from failures to errors so failures with zero errors in the window still appear with null error columns rather than being dropped.

**Edge cases:** if fewer than three distinct error codes exist in the window, the remaining rank columns are null-filled. Failures with no errors at all in the 24-hour window are retained in the output with all three columns null.

In [10]:
query = """
with failure_codes as 
                    (
                    select df.machineid,
                    df.datetime as failure_datetime,
                    df.failure as failure_component,
                    de.errorid,
                    count(*) as error_count
                    from df_failures df 
                    left join df_errors de 
                    on df.machineid = de.machineid
                    and de.datetime >= df.datetime - Interval 24 hour
                    and de.datetime < df.datetime
                    where de.errorid is not null
                    group by df.machineid , df.datetime , df.failure , de.errorid
                    ),
                    failure_rank as 
                    (
                    select *,
                    row_number() over(partition by machineid,failure_datetime 
                    order by error_count desc, errorid) as error_rank
                    from failure_codes
                    )
                    select machineid,
                    failure_datetime,
                    failure_component,
                    max(case when error_rank = 1 then errorid end) as rank_error_1,
                    max(case when error_rank = 2 then errorid end) as rank_error_2,
                    max(case when error_rank = 3 then errorid end) as rank_error_3
                    from failure_rank
                    group by machineid,failure_datetime,failure_component
                    order by machineid,failure_datetime
"""

q5 = conn.execute(query).fetch_df()
q5.head(20).style.hide(axis="index")

machineID,failure_datetime,failure_component,rank_error_1,rank_error_2,rank_error_3
1,2015-01-05 06:00:00,comp4,error5,nan,nan
1,2015-03-06 06:00:00,comp1,error1,nan,nan
1,2015-04-20 06:00:00,comp2,error2,error3,nan
1,2015-06-19 06:00:00,comp4,error5,nan,nan
1,2015-09-02 06:00:00,comp4,error5,nan,nan
1,2015-10-17 06:00:00,comp2,error2,error3,nan
1,2015-12-16 06:00:00,comp4,error5,nan,nan
2,2015-03-19 06:00:00,comp2,nan,error1,error2
2,2015-03-19 06:00:00,comp1,error1,nan,nan
2,2015-04-18 06:00:00,comp2,error2,error3,nan


## Part 2 — Data Pipeline

Design and partially implement a pipeline that joins the five tables, engineers features for the failure
prediction model, scores each machine-hour, and writes the results to a structured output.

### Unified join

In [11]:
for df in [df_telemetry, df_errors, df_failures, df_maint]:
    df['datetime'] = pd.to_datetime(df['datetime']).dt.floor('h')

In [12]:
conn.register('telemetry', df_telemetry)
conn.register('errors',    df_errors)
conn.register('failures',  df_failures)
conn.register('maint',     df_maint)
conn.register('machines',  df_machines)

In [13]:
tables_joined_sql = """
with telemetry_agg as ( 
    select machineID,
           datetime as datetime_hour,
           avg(volt)      as volt,
           avg(rotate)    as rotate,
           avg(pressure)  as pressure,
           avg(vibration) as vibration
    from telemetry
    group by machineID, datetime
),

errors_agg as (
    select machineID,
           datetime as datetime_hour,
           count(distinct errorID) as distinct_error_types
    from errors
    group by machineID, datetime
),

failures_agg as (
    select machineID,
           datetime as datetime_hour,
           1 as is_failure,
           max(failure) as failure_component
    from failures
    group by machineID, datetime
),

maint_agg as (
    select machineID,
           datetime as datetime_hour,
           count(*) as maint_count,
           max(comp) as last_maint_comp
    from maint
    group by machineID, datetime
)

select t.machineID,
       t.datetime_hour,
       t.volt,
       t.rotate,
       t.pressure,
       t.vibration,
       m.model,
       m.age,
       coalesce(e.distinct_error_types, 0) as distinct_error_types,
       coalesce(f.is_failure, 0)           as is_failure,
       f.failure_component,
       coalesce(mt.maint_count, 0)         as maint_count,
       mt.last_maint_comp
from telemetry_agg t
left join machines     m  on t.machineID = m.machineID
left join errors_agg   e  on t.machineID = e.machineID and e.datetime_hour = t.datetime_hour
left join failures_agg f  on t.machineID = f.machineID and f.datetime_hour = t.datetime_hour
left join maint_agg    mt on t.machineID = mt.machineID and mt.datetime_hour = t.datetime_hour
order by t.machineID, t.datetime_hour
"""

unified = conn.execute(tables_joined_sql).fetch_df()
unified.head(20).style.hide(axis="index")

machineID,datetime_hour,volt,rotate,pressure,vibration,model,age,distinct_error_types,is_failure,failure_component,maint_count,last_maint_comp
1,2015-01-01 06:00:00,176.217853,418.504078,113.077935,45.087686,model3,18,0,0,nan,0,nan
1,2015-01-01 07:00:00,162.879223,402.747490,95.460525,43.413973,model3,18,0,0,nan,0,nan
1,2015-01-01 08:00:00,170.989902,527.349825,75.237905,34.178847,model3,18,0,0,nan,0,nan
1,2015-01-01 09:00:00,162.462833,346.149335,109.248561,41.122144,model3,18,0,0,nan,0,nan
1,2015-01-01 10:00:00,157.610021,435.376873,111.886648,25.990511,model3,18,0,0,nan,0,nan
1,2015-01-01 11:00:00,172.504839,430.323362,95.927042,35.655017,model3,18,0,0,nan,0,nan
1,2015-01-01 12:00:00,156.556031,499.071623,111.755684,42.753920,model3,18,0,0,nan,0,nan
1,2015-01-01 13:00:00,172.522781,409.624717,101.001083,35.482009,model3,18,0,0,nan,0,nan
1,2015-01-01 14:00:00,175.324524,398.648781,110.624361,45.482287,model3,18,0,0,nan,0,nan
1,2015-01-01 15:00:00,169.218423,460.850670,104.848230,39.901735,model3,18,0,0,nan,0,nan


### Feature Engineering

Engineer Features 
Engineer at minimum: rolling 24-hour mean and standard deviation per sensor, hours since last
maintenance per component, error count in the past 12 hours.

In [14]:
unified = unified.sort_values(['machineID', 'datetime_hour']).reset_index(drop=True)

sensors = ['volt', 'rotate', 'pressure', 'vibration']


# Feature 1: Rolling 24h mean and std (shift(1) prevents leakage)
for sensor in sensors:
    shifted = unified.groupby('machineID')[sensor].shift(1)
    unified[f'{sensor}_24h_mean'] = shifted.groupby(unified['machineID']).transform(lambda x: x.rolling(24, min_periods=1).mean())
    unified[f'{sensor}_24h_std']  = shifted.groupby(unified['machineID']).transform(lambda x: x.rolling(24, min_periods=1).std())

# Feature 2: Hours since last maintenance per component
# pulling components dynamically so the pipeline doesn't break if a new component is added
components = df_maint['comp'].unique()

for comp in components:
    unified[f'maint_{comp}'] = (unified['last_maint_comp'] == comp).astype(int)
    unified[f'hours_since_{comp}'] = (
        unified.groupby('machineID')[f'maint_{comp}']
        .transform(lambda x: x.cumsum())
    )
    unified.drop(columns=[f'maint_{comp}'], inplace=True)
# average across all components into a single feature for the scoring cell
unified['hours_since_maint'] = unified[[f'hours_since_{comp}' for comp in components]].mean(axis=1)

# # Old approach — iterrows() inside a loop, O(n²), too slow for production
# components = ['comp1', 'comp2', 'comp3', 'comp4']
# for comp in components:
#     comp_df = df_maint[df_maint['comp'] == comp][['machineID', 'datetime']]
#     hrs = []
#     for _, row in unified.iterrows():
#         past = comp_df[
#             (comp_df['machineID'] == row['machineID']) &
#             (comp_df['datetime']  <  row['datetime_hour'])
#         ]['datetime']
#         hrs.append(
#             round((row['datetime_hour'] - past.max()).total_seconds() / 3600, 2)
#             if len(past) > 0 else np.nan
#         )
#     unified[f'hours_since_last_{comp}'] = hrs

# Feature 3 Error count in past 12 hours
unified = unified.sort_values(['machineID', 'datetime_hour']).reset_index(drop=True)

unified['errors_12h'] = (
    unified.groupby('machineID')['distinct_error_types']
    .transform(lambda x: x.shift(1).rolling(12, min_periods=1).sum())
)

print(f'Features added. Shape: {unified.shape}')
unified.head(5).style.hide(axis="index")

Features added. Shape: (876100, 27)


machineID,datetime_hour,volt,rotate,pressure,vibration,model,age,distinct_error_types,is_failure,failure_component,maint_count,last_maint_comp,volt_24h_mean,volt_24h_std,rotate_24h_mean,rotate_24h_std,pressure_24h_mean,pressure_24h_std,vibration_24h_mean,vibration_24h_std,hours_since_comp2,hours_since_comp4,hours_since_comp3,hours_since_comp1,hours_since_maint,errors_12h
1,2015-01-01 06:00:00,176.217853,418.504078,113.077935,45.087686,model3,18,0,0,nan,0,nan,nan,nan,nan,nan,nan,nan,nan,nan,0,0,0,0,0.000000,nan
1,2015-01-01 07:00:00,162.879223,402.747490,95.460525,43.413973,model3,18,0,0,nan,0,nan,176.217853,nan,418.504078,nan,113.077935,nan,45.087686,nan,0,0,0,0,0.000000,0.000000
1,2015-01-01 08:00:00,170.989902,527.349825,75.237905,34.178847,model3,18,0,0,nan,0,nan,169.548538,9.431836,410.625784,11.141591,104.269230,12.457390,44.250829,1.183494,0,0,0,0,0.000000,0.000000
1,2015-01-01 09:00:00,162.462833,346.149335,109.248561,41.122144,model3,18,0,0,nan,0,nan,170.028993,6.721032,449.533798,67.849599,94.592122,18.934956,40.893502,5.874970,0,0,0,0,0.000000,0.000000
1,2015-01-01 10:00:00,157.610021,435.376873,111.886648,25.990511,model3,18,0,0,nan,0,nan,168.137453,6.665324,423.687682,75.770259,98.256232,17.109194,40.950662,4.798255,0,0,0,0,0.000000,0.000000


### Model Scoring

Score with the pre-trained model; append predicted failure probability and binary prediction to
each row.

In [15]:
from sklearn.preprocessing import MinMaxScaler

feature_cols = [f'{s}_24h_mean' for s in sensors] + [f'{s}_24h_std' for s in sensors] + ['distinct_error_types', 'hours_since_maint', 'age']

feature_df = unified[feature_cols].fillna(0)
scaler = MinMaxScaler()
scaled = scaler.fit_transform(feature_df)

unified['risk_score'] = scaled.mean(axis=1)

print(unified['risk_score'].describe())
unified[['machineID', 'datetime_hour', 'risk_score', 'is_failure']].head(10).style.hide(axis="index")

count    876100.000000
mean          0.470234
std           0.038138
min           0.000000
25%           0.443686
50%           0.470627
75%           0.497461
max           0.641398
Name: risk_score, dtype: float64


machineID,datetime_hour,risk_score,is_failure
1,2015-01-01 06:00:00,0.081818,0
1,2015-01-01 07:00:00,0.352515,0
1,2015-01-01 08:00:00,0.412854,0
1,2015-01-01 09:00:00,0.488154,0
1,2015-01-01 10:00:00,0.476764,0
1,2015-01-01 11:00:00,0.487104,0
1,2015-01-01 12:00:00,0.472038,0
1,2015-01-01 13:00:00,0.472813,0
1,2015-01-01 14:00:00,0.463557,0
1,2015-01-01 15:00:00,0.463198,0


### Output — write to Parquet


In [16]:
import os

unified['year']  = unified['datetime_hour'].dt.year
unified['month'] = unified['datetime_hour'].dt.month

output_base = 'output/scored'

for (model, year, month), group in unified.groupby(['model', 'year', 'month']):
    path = f'{output_base}/model={model}/year={year}/month={month:02d}'
    os.makedirs(path, exist_ok=True)
    group.drop(columns=['model', 'year', 'month']).to_parquet(f'{path}/data.parquet', index=False)

print('Done. Parquet files written to output/scored/')

Done. Parquet files written to output/scored/


### Q2.1 — Data integrity across time

Your features are built from historical maintenance and error records. How do you ensure that when you
score a row for time T, none of the features contain information from after time T? Walk through one
specific feature and how you'd enforce this.

**What it does:** explains how the pipeline prevents feature leakage, ensuring no feature for time T contains information from after time T.

**Why this approach:** the main risk is using data from after time T to compute a feature for time T. For the rolling 24-hour mean of volt, I enforce this by calling shift(1) before applying the rolling window. shift(1) moves each value forward by one position so the current hour's reading is excluded from its own rolling average. Without this, the model would be trained on features that include information it wouldn't have at scoring time, which inflates performance metrics and causes silent failures in production.

**Edge cases:** the same shift(1) pattern is applied to all sensor rolling features and the 12-hour error count. For hours_since_last_maintenance, leakage is prevented by design, cumsum() counts forward from each maintenance flag so only past events are included.

## Part 3 — Data Quality Checks

### DQ1 — Referential integrity

In [17]:
dq1_sql = """
select 'errors'   as source_table, count(*) as orphaned_rows from df_errors   where machineID not in (select machineID from df_machines)
union all
select 'failures' as source_table, count(*) as orphaned_rows from df_failures where machineID not in (select machineID from df_machines)
union all
select 'maint'    as source_table, count(*) as orphaned_rows from df_maint    where machineID not in (select machineID from df_machines)
"""

conn.execute(dq1_sql).fetch_df()

,source_table,orphaned_rows
0,errors,0
1,failures,0
2,maint,0


### DQ2 — Completeness check

Telemetry should have one row per machine per hour. Building the expected hourly grid
and comparing against what's actually there.

In [18]:
dq2_sql = """
with time_spine as (
    select generate_series as hour
    from generate_series(
        (select min(datetime) from df_telemetry),
        (select max(datetime) from df_telemetry),
        interval '1 hour'
    )
),
expected as (
    select m.machineID, ts.hour
    from df_machines m
    cross join time_spine ts
),
actual as (
    select machineID, datetime as hour from df_telemetry
)
select count(*) as missing_hourly_rows
from expected e
left join actual a on e.machineID = a.machineID and e.hour = a.hour
where a.machineID is null
"""

conn.execute(dq2_sql).fetch_df()

,missing_hourly_rows
0,0


### Q3.1 — Monitoring a heavily imbalanced pipeline

**Question:** Q3.1 Monitoring a heavily imbalanced pipeline
This dataset has 136 failure events out of 876,100 rows. A model that predicts 'no failure' for every row
would be correct 99.98% of the time. How would your observability framework detect if the pipeline were
silently producing that kind of degenerate output? What metric or signal would you monitor, and why?



**Answer** — Accuracy is the wrong metric here. 136 failures out of 876,100 rows means a model that never predicts failure is still 99.98% accurate, so accuracy tells you nothing. Here's what I'd actually monitor:


1. Score distribution
Every run I'd pull the P90, P95, and MAX of the risk scores across all machines. A working model should show some machines scoring noticeably higher than others. You'd expect a tail above 0.3 for machines that are genuinely at risk. If everything collapses to near zero with no variation, the model has stopped differentiating. I'd alert if MAX stays below 0.1 across an entire run, that's a sign the pipeline is silently broken even though it looks like it's running fine.
2. Positive prediction count
136 failures over a year is roughly 0.37 per day. So I'd track how many machines get flagged as high risk each day and alert if we see zero flags for more than 48 consecutive hours. That said, low scores don't always mean something's wrong. If machines are on planned downtime or sensor readings look genuinely stable, low scores make sense. So I'd cross check this against whether machines are actually running before raising an alarm.
3. Retrospective recall on known failures
I'd keep a small holdout of past confirmed failures and after every production run check whether the model assigned high risk scores to those machines in the hours leading up to those known events. If recall on that holdout drops below 0.5, the model has degraded. This is the most direct check. You already know what should have been flagged, so you just verify the model would have caught it.

## Bonus — Streamlit Data Quality Dashboard

Built a minimal Streamlit app showing referential integrity results, telemetry completeness,
failure distribution by model, and risk score distribution.

In [19]:
%%writefile app.py
import streamlit as st
import pandas as pd
import duckdb
import plotly.express as px

st.set_page_config(page_title="PdM Data Quality Summary", layout="wide")
st.title("Predictive Maintenance — Data Quality Summary")

df_telemetry = pd.read_csv('data/PdM_telemetry.csv')
df_errors    = pd.read_csv('data/PdM_errors.csv')
df_failures  = pd.read_csv('data/PdM_failures.csv')
df_maint     = pd.read_csv('data/PdM_maint.csv')
df_machines  = pd.read_csv('data/PdM_machines.csv')

for df in [df_telemetry, df_errors, df_failures, df_maint]:
    df['datetime'] = pd.to_datetime(df['datetime']).dt.floor('h')

conn = duckdb.connect()
conn.register('df_telemetry', df_telemetry)
conn.register('df_errors',    df_errors)
conn.register('df_failures',  df_failures)
conn.register('df_maint',     df_maint)
conn.register('df_machines',  df_machines)

st.subheader("Referential Integrity")
dq1 = conn.execute("""
    select 'errors'   as source_table, count(*) as orphaned_rows from df_errors   where machineID not in (select machineID from df_machines)
    union all
    select 'failures' as source_table, count(*) as orphaned_rows from df_failures where machineID not in (select machineID from df_machines)
    union all
    select 'maint'    as source_table, count(*) as orphaned_rows from df_maint    where machineID not in (select machineID from df_machines)
""").fetch_df()
st.dataframe(dq1, use_container_width=True)
if dq1['orphaned_rows'].sum() == 0:
    st.success("No referential integrity violations found across all tables.")
else:
    st.error("Referential integrity violations detected.")

st.subheader("Telemetry Completeness")
dq2 = conn.execute("""
    with time_spine as (
        select generate_series as hour
        from generate_series(
            (select min(datetime) from df_telemetry),
            (select max(datetime) from df_telemetry),
            interval '1 hour'
        )
    ),
    expected as (
        select m.machineID, ts.hour
        from df_machines m
        cross join time_spine ts
    ),
    actual as (
        select machineID, datetime as hour from df_telemetry
    )
    select
        count(*) as total_expected,
        sum(case when a.machineID is null then 1 else 0 end) as missing_rows,
        round(sum(case when a.machineID is null then 1 else 0 end) * 100.0 / count(*), 4) as pct_missing
    from expected e
    left join actual a on e.machineID = a.machineID and e.hour = a.hour
""").fetch_df()
col1, col2, col3 = st.columns(3)
col1.metric("Total expected rows", f"{dq2['total_expected'][0]:,}")
col2.metric("Missing rows",        f"{dq2['missing_rows'][0]:,}")
col3.metric("% missing",           f"{dq2['pct_missing'][0]}%")

st.subheader("Failure Distribution by Model")
failure_dist = conn.execute("""
    select m.model, count(*) as failures
    from df_failures f
    join df_machines m on f.machineID = m.machineID
    group by m.model
    order by failures desc
""").fetch_df()
fig = px.bar(failure_dist, x='model', y='failures', title='Failures by machine model')
st.plotly_chart(fig, use_container_width=True)

Overwriting app.py


In [20]:
subprocess.Popen(['streamlit', 'run', 'app.py'])
print("Streamlit running at http://localhost:8501")

Streamlit running at http://localhost:8501


2026-05-02 16:43:10.565 Uvicorn server started on 0.0.0.0:8502



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8502
  Network URL: http://192.168.1.68:8502

  For better performance, install the Watchdog module:

  $ xcode-select --install
  $ pip install watchdog
            


2026-05-02 16:43:15.194 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-05-02 16:43:15.313 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
